# cancel-risk: dự báo huỷ booking và sai số số phòng bán vượt

Notebook chạy cục bộ, lấy dữ liệu thẳng từ generator (`edai1/generator`), không cần cluster. Bản chạy trên cluster là
`ml/training_pipeline.py` (Feast `get_historical_features` + view `gold.labels_cancel`); ở đây dựng lại đúng các input
của model từ bảng booking của generator để:

1. train XGBoost với split theo thời gian (không random), đo AUC so với trần AUC của nhãn;
2. so xác suất dự báo với **xác suất huỷ thật** `p_cancel_true` mà generator biết (Brier, reliability);
3. đo sai số số phòng bán vượt K khi dùng xác suất model thay cho xác suất thật (novelty edai-2, Idea 1);
4. xem drift từ 2025-06-01 làm lệch calibration thế nào, và retrain trên dữ liệu gần đây sửa được không.

Thu nhỏ dims (2.000 khách sạn, 100.000 khách) để chạy vài phút; tỉ lệ huỷ, hệ số nhãn và drift giữ nguyên config.

In [1]:
import sys
sys.path[:0] = ["../edai1", "../edai2/mcp_server"]
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

from generator.offline_gen import gen_bookings, gen_dims, load_config
from overbooking import limit

cfg = load_config("../edai1/generator/config.yaml")
cfg["dims"] = {"hotels": 2000, "customers": 100_000}
cfg["shards_per_day"] = 4
dims = gen_dims(cfg)
DAYS = list(range(30, 240, 3))             # tháng 2 -> 8/2025, cắt qua mốc drift 2025-06-01 (ngày 151)
b = pd.concat([gen_bookings(cfg, dims, d, s) for d in DAYS for s in range(cfg["shards_per_day"])], ignore_index=True)
b = b.sort_values(["created_at", "booking_id"]).drop_duplicates("idempotency_key")      # = dedup ở Silver
len(b), b["status"].eq("cancelled").mean().round(3)

(1072412, np.float64(0.392))

## Input của model

Giống `MODEL_INPUTS` trừ các feature cửa sổ 7d/30d/90d của Gold (cần DP3): `lead_days`, `nights`, `free_cancel`,
`loyalty_tier`, và lịch sử khách tính point-in-time chỉ từ booking **trước** lúc đặt (`prior_bookings`,
`has_customer_history`). Nhãn = huỷ trước ngày nhận phòng (view `labels_cancel`).

In [2]:
b["lead_days"] = (b["checkin"] - b["created_at"]).dt.days.clip(lower=0)
b["free_cancel"] = (b["cancellation_policy"] == "flexible").astype(int)
b["loyalty_tier"] = dims.customers.set_index("customer_id").loc[b["customer_id"], "loyalty_tier"].values
b["prior_bookings"] = b.groupby("customer_id").cumcount()                     # chỉ booking trước đó của khách
b["has_customer_history"] = (b["prior_bookings"] > 0).astype(int)
b["y"] = (b["status"] == "cancelled").astype(int)
b["day"] = (b["created_at"] - pd.Timestamp(cfg["start_date"])).dt.days
INPUTS = ["lead_days", "nights", "free_cancel", "loyalty_tier", "prior_bookings", "has_customer_history"]
DRIFT = (pd.Timestamp(cfg["label"]["drift"]["date"]) - pd.Timestamp(cfg["start_date"])).days
pre, post = b[b["day"] < DRIFT], b[b["day"] >= DRIFT]
pre.groupby(pre["day"] // 30)[["y", "lead_days"]].mean().round(3).T

day,1,2,3,4,5
y,0.329,0.333,0.327,0.332,0.325
lead_days,18.597,18.740,18.462,19.032,19.025


## Train v1 trước drift, split theo thời gian 80/20

In [3]:
cut = pre["created_at"].quantile(0.8)
train, valid = pre[pre["created_at"] <= cut], pre[pre["created_at"] > cut]
v1 = XGBClassifier(n_estimators=300, max_depth=6, eval_metric="auc").fit(train[INPUTS], train["y"])
p = v1.predict_proba(valid[INPUTS])[:, 1]

def report(name, y, p, p_true):
    return {"tập": name, "AUC": roc_auc_score(y, p), "AUC của p thật (trần)": roc_auc_score(y, p_true),
            "huỷ dự báo": p.mean(), "huỷ thật": y.mean(), "lệch calibration": abs(p.mean() - y.mean()),
            "Brier vs nhãn": brier_score_loss(y, p), "Brier vs p thật": np.mean((p - p_true) ** 2)}

pd.DataFrame([report("validation v1 (trước drift)", valid["y"], p, valid["p_cancel_true"])]).round(4)

,tập,AUC,AUC của p thật (trần),huỷ dự báo,huỷ thật,lệch calibration,Brier vs nhãn,Brier vs p thật
0,validation v1 (trước drift),0.6718,0.7311,0.3312,0.3322,0.0009,0.2048,0.016


AUC của model thấp hơn trần (AUC của chính `p_cancel_true`) vì nhãn có thành phần ẩn `propensity` mà model không
thấy. Cổng promote vì vậy dùng sàn `MIN_AUC` 0,60 + lệch calibration ≤ 0,03, không dùng ngưỡng AUC cao.

## Reliability: xác suất dự báo so với xác suất thật theo decile

In [4]:
valid = valid.assign(p=p, bin=pd.qcut(p, 10, labels=False, duplicates="drop"))
valid.groupby("bin").agg(p_du_bao=("p", "mean"), p_that=("p_cancel_true", "mean"), ty_le_huy=("y", "mean"),
                         n=("y", "size")).round(3)

,p_du_bao,p_that,ty_le_huy,n
bin,,,,
0,0.098,0.123,0.126,12271
1,0.166,0.178,0.178,12271
2,0.210,0.219,0.227,12275
3,0.247,0.250,0.248,12267
4,0.290,0.287,0.287,12272
5,0.356,0.358,0.358,12269
6,0.416,0.416,0.416,12279
7,0.465,0.466,0.469,12264
8,0.500,0.496,0.493,12270


## Sai số số phòng bán vượt K

K tính bằng `mcp_server/overbooking.limit` (Poisson-binomial, c_spoil = 1, c_walk = 3) cho từng ô hotel × đêm nhận
phòng, một lần với xác suất model, một lần với xác suất thật. Sai số K là cái revenue manager thực sự chịu.

In [5]:
def k_error(df, probs):
    df = df.assign(p_model=probs)
    rows = []
    for (h, day), g in df.groupby(["hotel_id", df["checkin"].dt.normalize()]):
        if len(g) >= 5:
            rows.append((len(g), limit(g["p_model"].tolist(), 1, 3), limit(g["p_cancel_true"].tolist(), 1, 3)))
    k = pd.DataFrame(rows, columns=["bookings", "k_model", "k_true"])
    return {"số ô": len(k), "K thật TB": k["k_true"].mean(), "K model TB": k["k_model"].mean(),
            "|K model - K thật| TB": (k["k_model"] - k["k_true"]).abs().mean(),
            "tỉ lệ ô bán vượt quá tay (K model > K thật)": (k["k_model"] > k["k_true"]).mean()}

pd.DataFrame([k_error(valid, p)], index=["validation v1"]).round(3)

,số ô,K thật TB,K model TB,|K model - K thật| TB,tỉ lệ ô bán vượt quá tay (K model > K thật)
validation v1,3976,3.27,3.252,0.304,0.136


## Sau drift: model v1 lệch calibration, retrain trên dữ liệu gần đây

Từ 2025-06-01 generator đổi hệ số lead time của nhãn (concept drift) và kéo dài lead time ×1,5 (covariate drift).
`drift_monitor` bắt bằng PSI trên `lead_days` và lệch calibration > 0,03 rồi gọi retrain.

In [6]:
post = post.sort_values("created_at")
cut2 = post["created_at"].quantile(0.5)
post_train, post_valid = post[post["created_at"] <= cut2], post[post["created_at"] > cut2]
# v2 chỉ train trên cửa sổ gần đây (sau drift), như drift_monitor gọi retrain với since/until mới
v2 = XGBClassifier(n_estimators=300, max_depth=6, eval_metric="auc").fit(post_train[INPUTS], post_train["y"])
p_v1, p_v2 = v1.predict_proba(post_valid[INPUTS])[:, 1], v2.predict_proba(post_valid[INPUTS])[:, 1]
pd.DataFrame([report("sau drift, model v1", post_valid["y"], p_v1, post_valid["p_cancel_true"]),
              report("sau drift, model v2 (retrain)", post_valid["y"], p_v2, post_valid["p_cancel_true"])]).round(4)

,tập,AUC,AUC của p thật (trần),huỷ dự báo,huỷ thật,lệch calibration,Brier vs nhãn,Brier vs p thật
0,"sau drift, model v1",0.6721,0.7438,0.3477,0.4745,0.1268,0.2447,0.0402
1,"sau drift, model v2 (retrain)",0.6864,0.7438,0.4737,0.4745,0.0008,0.2239,0.0194


In [7]:
sys.path.insert(0, "../edai2/dags")
from drift_stats import psi                                    # cùng hàm DAG drift_monitor dùng
pd.DataFrame({"PSI lead_days so với tập train v1": [psi(train["lead_days"].values, pre[pre["created_at"] > cut]["lead_days"].values),
                                                    psi(train["lead_days"].values, post["lead_days"].values)]},
             index=["trước drift", "sau drift"]).round(3)

,PSI lead_days so với tập train v1
trước drift,0.002
sau drift,0.247


In [8]:
pd.DataFrame([k_error(post_valid, p_v1), k_error(post_valid, p_v2)], index=["sau drift v1", "sau drift v2"]).round(3)

,số ô,K thật TB,K model TB,|K model - K thật| TB,tỉ lệ ô bán vượt quá tay (K model > K thật)
sau drift v1,7806,5.132,3.513,1.624,0.002
sau drift v2,7806,5.132,5.114,0.356,0.161


## Kết luận

- AUC của model nằm dưới trần của nhãn (thành phần ẩn), nên cổng promote dựa vào calibration và so với production.
  Trước drift, xác suất model khớp xác suất thật theo từng decile, sai số K trung bình dưới nửa phòng.
- Sau drift, v1 dự báo tỉ lệ huỷ thấp hơn thực tế rõ rệt (lệch calibration vượt 0,03) và PSI `lead_days` vượt 0,2:
  đúng hai tín hiệu `drift_monitor` dùng để gọi retrain. K của v1 thấp hơn K thật: bán vượt quá ít, phòng trống.
- Retrain chỉ trên cửa sổ sau drift kéo lệch calibration và sai số K xuống (xem bảng trên). Vì overbooking dùng thẳng
  xác suất, calibration quan trọng hơn AUC.
- Giới hạn: notebook dùng kết cục đã biết của booking sau drift; pipeline thật chỉ có nhãn khi đã qua ngày nhận phòng
  (`LABEL_SQL` lấy booking tới `until - 90` ngày), nên `drift_monitor` chờ 30 ngày booking sau mốc phát hiện + 90 ngày
  cho nhãn chín (`drift_stats.retrain_window`, ~4 tháng) rồi mới retrain với `since` = mốc phát hiện.
- Số trong notebook đổi theo mẫu (dims thu nhỏ); số chính thức của generator nằm trong `profile_data`.